# Huggingface with langchain

Hugging Face is a popular AI platform and community used to build, train, and deploy machine learning models, especially for Natural Language Processing (NLP), vision, and audio tasks, by providing thousands of pre-trained models, datasets, and tools like the transformers library, simplifying complex AI development for everyone from beginners to experts.

Package install

pip install langchain

pip install langchain-huggingface

pip install huggingface_hub

In [1]:
import os
os.environ['huggingface_token']="add your huggingface token here"

In [2]:
from langchain_core.prompts import PromptTemplate

template = "Tell me about {topic}"
prompt = PromptTemplate.from_template(template)

print(prompt.format(topic="AI"))


Tell me about AI


In [3]:
import os
from openai import OpenAI

client = OpenAI(
    #provider=together",
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["huggingface_token"],
)

completion = client.chat.completions.create(
    model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[
        {
            "role": "user",
            "content": "Summarize the plot of 'Matrix'."
        }
    ],
)

print(completion.choices[0].message.content)

*The Matrix* (1999) follows Thomas Anderson, a computer hacker known as Neo, who discovers that the world he knows is a simulated reality created by sentient machines to subdue humanity while their bodies are used as an energy source. Freed by a group of rebels led by Morpheus and the warrior Trinity, Neo learns to manipulate the Matrix and embrace his role as “The One,” a prophesied savior. After battling the relentless Agent Smith and making a self-sacrificing choice to save Trinity, Neo dies and resurrects within the Matrix, gaining powers that let him defeat the Agents. The film ends with Neo declaring to the machines that he will free humanity, setting the stage for revolution.


In [4]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    api_key=os.environ["huggingface_token"],
)

image = client.text_to_image(
    "A cute alien creature in a spaceship",
    model="stabilityai/stable-diffusion-xl-base-1.0"
)

#image.save("output.png")
print(image)


<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1024x1024 at 0x11C838590>


In [23]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Create endpoint LLM
endpoint = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["huggingface_token"],
    task="conversational",   # VERY IMPORTANT
    temperature=0.7,
)

# Step 2: Wrap into Chat model
llm = ChatHuggingFace(llm=endpoint)

# Step 3: Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain {topic} in simple terms")
])

chain = prompt | llm

response = chain.invoke({"topic": "mahadev"})

print(response.content)

Think of Mahadev as “the big boss of letting go.”

1. **Who is he?**  
   - Common name: Shiva.  
   - Title: Maha-dev = “the greatest god.”

2. **What does he do?**  
   - When the world gets too messy, he presses the “reset button” by dancing (the cosmic dance).  
   - He teaches that you can drop bad habits, anger, or even big illusions—just let them go.  
   - He sits quietly in meditation most of the time, showing inner calm is possible.

3. **How do people picture him?**  
   - Holds a small drum (sound of creation) and fire (destruction).  
   - Third eye on the forehead = extra-powerful insight.  
   - Has a snake round his neck and the river Ganga flows from his hair—symbols that even danger and chaos can be handled peacefully.

4. **Why do many like him?**  
   - He accepts everyone—gods, demons, humans—no VIP list.  
   - You can reach him with just a sincere heart; no fancy rituals required.

In one line: Mahadev is the cool, calm god who shows how to destroy life’s junk so

In [25]:
#prompt = PromptTemplate(
#    input_variables=["product"]
 #   template="what is good name for the company that make {product}"
#)

In [26]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# 1️⃣ Create HF Endpoint (LLM)
llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["huggingface_token"],
    task="conversational",  # IMPORTANT
    temperature=0.7,
    max_new_tokens=512,
)

# 2️⃣ Wrap inside ChatHuggingFace
chat_model = ChatHuggingFace(llm=llm)

# 3️⃣ Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher."),
    ("human", """
Generate {no of qus} multiple choice questions about {topic}.

Return ONLY valid JSON format.
""")
])

# 4️⃣ Chain
chain = prompt | chat_model | StrOutputParser()

response = chain.invoke({
    "no of qus":10,
    "topic":"sanatan"
})

print(response)

```json
[
  {
    "question": "What does the term 'Sanatan Dharma' literally mean?",
    "options": [
      "Eternal Religion",
      "Modern Faith",
      "New Path",
      "Temporary Belief"
    ],
    "answer": "Eternal Religion"
  },
  {
    "question": "Which ancient text is considered one of the foundational scriptures of Sanatan Dharma?",
    "options": [
      "The Bible",
      "The Quran",
      "The Rig Veda",
      "The Tao Te Ching"
    ],
    "answer": "The Rig Veda"
  },
  {
    "question": "Which deity is known as the 'Preserver' in the Hindu Trinity of Sanatan Dharma?",
    "options": [
      "Brahma",
      "Vishnu",
      "Shiva",
      "Indra"
    ],
    "answer": "Vishnu"
  },
  {
    "question": "What is the ultimate goal of life according to Sanatan Dharma?",
    "options": [
      "Accumulation of Wealth",
      "Political Power",
      "Moksha (Liberation)",
      "Fame"
    ],
    "answer": "Moksha (Liberation)"
  },
  {
    "question": "Which epic narrates th

In [42]:
import os
import json
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1️⃣ Create HuggingFaceEndpoint with conversational task
hf_llm = HuggingFaceEndpoint(
    repo_id="moonshotai/Kimi-K2-Instruct-0905",
    huggingfacehub_api_token=os.environ["huggingface_token"],
    task="conversational",   # VERY IMPORTANT
    temperature=0.7,
    max_new_tokens=512,
)

# 2️⃣ Wrap it inside ChatHuggingFace
llm = ChatHuggingFace(llm=hf_llm)

# 3️⃣ Chat Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher."),
    ("human", """
Generate {num_questions} multiple choice questions about {topic}.

Return ONLY valid JSON in this format:

[
  {{
    "question": "...",
    "options": ["A", "B", "C", "D"],
    "answer": "..."
  }}
]
""")
])

chain = prompt | llm | StrOutputParser()

response = chain.invoke({
    "topic": "Python basics",
    "num_questions": 5
})

print(response)

[
  {
    "question": "Which of the following is the correct way to create a list in Python?",
    "options": ["list = {}", "list = []", "list = ()", "list = //"],
    "answer": "list = []"
  },
  {
    "question": "What does the len() function return when called on a string?",
    "options": ["The number of characters", "The number of words", "The size in bytes", "The memory address"],
    "answer": "The number of characters"
  },
  {
    "question": "Which keyword is used to define a function in Python?",
    "options": ["def", "function", "define", "fun"],
    "answer": "def"
  },
  {
    "question": "What is the output of print(3 * '2')?",
    "options": ["6", "222", "32", "Error"],
    "answer": "222"
  },
  {
    "question": "Which of the following data types is mutable in Python?",
    "options": ["tuple", "str", "list", "int"],
    "answer": "list"
  }
]


In [33]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate

endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-2-7b-chat-hf",
    huggingfacehub_api_token=os.environ["huggingface_token"],
    task="chat-completion",
    temperature=0.7,
    max_new_tokens=512,
)

llm = ChatHuggingFace(llm=endpoint)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain {topic} in simple terms")
])

chain = prompt | llm

response = chain.invoke({"topic": "Mahadev"})
print(response.content)

BadRequestError: (Request ID: Root=1-69a13f3a-513bd73d2d6437f44126389a;14b71029-1752-4a13-8a9b-54d96c980506)

Bad request:
{'message': "The requested model 'meta-llama/Llama-2-7b-chat-hf' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}